<a href="https://colab.research.google.com/github/pizzeman/ds2002-fa26/blob/main/notebooks/02-sql-databases/2026-09-11%20%E2%80%94%20SQL%20Challenge%20Set%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [10]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [11]:
q('''
  SELECT tracks.title, artists.name, artists.country
  FROM tracks
  LEFT JOIN artists ON tracks.artist_id = artists.artist_id
  ORDER BY tracks.title
''')

,title,name,country
0,Aurora,Kestrel,UK
1,Coastline,The Blue Ridge,US
2,Foothills,The Blue Ridge,US
3,Nightfall,Kestrel,UK
4,Ridgeline,The Blue Ridge,US
5,Skyline,Nova Waves,US
6,Sol,Marisol,ES
7,Undertow,Nova Waves,US
8,Untitled Demo,Kestrel,UK


### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [12]:
q('''
  SELECT tracks.genre, AVG(tracks.seconds) AS avg_seconds
  FROM tracks
  GROUP BY tracks.genre
  ORDER BY avg_seconds DESC
''')

,genre,avg_seconds
0,Electronic,287.5
1,Pop,220.5
2,Latin,210.0
3,Folk,203.0
4,None,150.0


### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [13]:
q('''
  SELECT plays.user, COUNT(plays.play_id) AS plays, COUNT(DISTINCT plays.track_id) AS track_count
  FROM plays
  GROUP BY plays.user
''')

,user,plays,track_count
0,ava,4,4
1,ben,3,3
2,cara,2,2
3,dan,2,2


### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [14]:
q('''
  SELECT tracks.title, COUNT(plays.track_id) AS track_count
  FROM tracks
  LEFT JOIN plays ON tracks.track_id = plays.track_id
  GROUP BY tracks.title HAVING track_count == 0
  ORDER BY track_count DESC
''')

,title,track_count
0,Untitled Demo,0
1,Ridgeline,0


### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [15]:
q('''
  SELECT artists.name, SUM(tracks.seconds) AS seconds, ROUND(SUM(tracks.seconds) / 60.0, 1) AS mins
  FROM artists
  LEFT JOIN tracks ON artists.artist_id = tracks.artist_id
  GROUP BY artists.name
  ORDER BY mins DESC
''')

,name,seconds,mins
0,Kestrel,725,12.1
1,The Blue Ridge,609,10.2
2,Nova Waves,441,7.3
3,Marisol,210,3.5


### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [16]:
q('''
  SELECT tracks.track_id, tracks.title, tracks.genre
  FROM tracks
''')

,track_id,title,genre
0,10,Skyline,Pop
1,11,Undertow,Pop
2,12,Foothills,Folk
3,13,Aurora,Electronic
4,14,Nightfall,Electronic
5,15,Sol,Latin
6,16,Coastline,Folk
7,17,Ridgeline,Folk
8,18,Untitled Demo,None


If I were to add that WHERE command, all titles that do not have Pop genre and is not None would be shown. So, those with Pop and None under genre would be removed.

### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [17]:
q('''
  SELECT plays.played_on, COUNT(plays.track_id) AS num_of_plays, COUNT(DISTINCT plays.user) AS num_of_users
  FROM plays
  GROUP BY plays.played_on
  ORDER BY plays.played_on
''')

,played_on,num_of_plays,num_of_users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [18]:
assert len(q('''
  SELECT tracks.title, artists.name, artists.country
  FROM tracks
  LEFT JOIN artists ON tracks.artist_id = artists.artist_id
  ORDER BY tracks.title
''')) == 9, 'Q1 should return one row per track'
assert len(q('''
  SELECT tracks.title, COUNT(plays.track_id) AS track_count
  FROM tracks
  LEFT JOIN plays ON tracks.track_id = plays.track_id
  GROUP BY tracks.title HAVING track_count == 0
  ORDER BY track_count DESC
''')) == 2, 'Q4: two tracks have never been played'
assert q('''
  SELECT plays.user, COUNT(plays.play_id) AS plays, COUNT(DISTINCT plays.track_id) AS track_count
  FROM plays
  GROUP BY plays.user
''')['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

Total listening time was confusing in question 5. I had trouble figuring out how to round to one decimal place while performing the math. The solution I found was to divide by 60.0 instead of 60. This prevented my answer from always outputting .0 at the end of each value.